# MIA work-mode progress model with module adjustment

This notebook tests whether `mean_progress` differs between `playlist` and `zpdes` in the MIA-only data, while accounting for repeated observations from students, classrooms, and modules.

The model is fitted for two populations and for their pooled union:

1. `exclusive_modes`: students who appear in exactly one of `playlist` or `zpdes`.
2. `both_modes`: students who appear in both `playlist` and `zpdes`.
3. `combined`: all retained rows from the first two populations. Its playlist group contains playlist-only students plus playlist observations from both-mode students; its zpdes group is constructed in the same way.

There is no global minimum-exercise filter. Each contiguous work-mode run within a student's module-qualified activity must contain at least four unique first-attempt exercises. The combined model pools those retained activity-level sequence rows.

## 1. Setup

The notebook imports the reusable functions from `scripts/model_work_mode_progress.py`. This keeps the notebook readable and ensures the command-line script and notebook run the same analysis logic.

In [1]:
from __future__ import annotations

import importlib
import sys
import textwrap
from argparse import Namespace
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import scripts.model_work_mode_first_attempt_trajectory as first_attempt_model  # noqa: E402
import scripts.model_work_mode_progress as work_mode_model  # noqa: E402

importlib.reload(first_attempt_model)
importlib.reload(work_mode_model)

from scripts.model_work_mode_progress import (  # noqa: E402
    build_activity_level,
    build_forest_figure,
    build_top_module_plot_table,
    fit_clustered_ols_sensitivity,
    fit_half_success_model,
    fit_mixed_model,
    fit_population_interaction_model,
    load_attempts,
    split_populations,
    summarize_half_exercise_elo,
)

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)

## 2. Parameters

This notebook should be run on the MIA-only data, not the scoped `data_miaam` parquet files. The default below uses the local raw MIA parquet in `data_MIA/`.

`config_mia.json` activity `learning_items` are used to map playlist exercise IDs to UUID-backed activities and module labels, because playlist rows do not carry that hierarchy directly. Local ordinal arrays in `exo_mia.json` are not treated as global identifiers.

All playlist rows are included, including playlists spanning multiple modules. Activity-level contiguous sequences prevent those playlists from being treated as one cross-module progress sequence.

`FOREST_POPULATION = None` pools `exclusive_modes` and `both_modes` for both the forest-plot module ranking and its plotted estimates. The regression section still fits both source populations, their pooled union, and the population interaction.

In [2]:
INPUT_FILE = PROJECT_ROOT / "data_MIA" / "986-neurips-mia_20260415_100024.parquet"
EXERCISE_CATALOG_JSON = PROJECT_ROOT / "data_MIA" / "exo_mia.json"
MODULE_CONFIG_JSON = PROJECT_ROOT / "data_MIA" / "config_mia.json"
EXERCISE_ELO_FILE = (
    PROJECT_ROOT
    / "artifacts"
    / "sources"
    / "mia"
    / "artifacts"
    / "derived"
    / "agg_exercise_elo.parquet"
)

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"MIA-only input not found: {INPUT_FILE}. Update INPUT_FILE in this cell."
    )
if not EXERCISE_CATALOG_JSON.exists() or not MODULE_CONFIG_JSON.exists():
    raise FileNotFoundError("Missing MIA catalog JSON files in data_MIA/.")
if not EXERCISE_ELO_FILE.exists():
    raise FileNotFoundError(f"Missing calibrated MIA exercise Elo: {EXERCISE_ELO_FILE}")

OUTPUT_DIR = PROJECT_ROOT / "artifacts" / "regression_mia_notebook"

MIN_ACTIVITY_EXERCISES = 4
TOP_N_MODULES = 5
MAXITER = 200
KEEP_ONLY_SINGLE_MODULE_PLAYLISTS = False
ALLOW_SINGLE_MODE_TOP_MODULES = False
PLOT_BY_POPULATION = False
FOREST_POPULATION = None
RUN_MODEL = True

args = Namespace(
    input_file=INPUT_FILE,
    input_csv=None,
    data_dir=None,
    exercise_catalog_json=EXERCISE_CATALOG_JSON,
    module_config_json=MODULE_CONFIG_JSON,
    output_dir=OUTPUT_DIR,
    min_unique_exercises=None,
    min_activity_exercises=MIN_ACTIVITY_EXERCISES,
    top_n_modules=TOP_N_MODULES,
    maxiter=MAXITER,
    keep_only_single_module_playlists=KEEP_ONLY_SINGLE_MODULE_PLAYLISTS,
    allow_single_mode_top_modules=ALLOW_SINGLE_MODE_TOP_MODULES,
    plot_by_population=PLOT_BY_POPULATION,
    forest_population=FOREST_POPULATION,
    skip_model=not RUN_MODEL,
)

## 3. Load and standardize attempts

The loader standardizes columns to the names used by the analysis: `student_id`, `classroom_id`, `playlist_id`, `exercise_id`, `activity_id`, `module`, `created_at`, `success`, and `work_mode`.

If a `source` column exists, student ids and playlist ids are prefixed by source to avoid accidental collisions across AM/MIA sources.

Playlist rows are not restricted by the number of modules associated with their playlist id. Module-qualified activities and contiguous work-mode runs define the progress sequences.

In [3]:
attempts = load_attempts(args)
exercise_elo = pd.read_parquet(
    EXERCISE_ELO_FILE,
    columns=["exercise_id", "activity_id", "exercise_elo", "calibrated"],
)

playlist_contexts = None
max_modules_per_playlist_context = None
source_values = None
if "source" in attempts.columns:
    source_values = ", ".join(sorted(attempts["source"].dropna().astype(str).unique()))

if "playlist_id" in attempts.columns:
    playlist_context_modules = (
        attempts[attempts["work_mode"] == "playlist"]
        .groupby("playlist_id")["module"]
        .nunique()
    )
    playlist_contexts = playlist_context_modules.size
    max_modules_per_playlist_context = (
        None if playlist_context_modules.empty else int(playlist_context_modules.max())
    )

attempt_summary = pd.DataFrame(
    [
        {
            "attempt_rows": len(attempts),
            "students": attempts["student_id"].nunique(),
            "classrooms": attempts["classroom_id"].nunique(),
            "modules": attempts["module"].nunique(),
            "activities": attempts[["module", "activity_id"]].drop_duplicates().shape[0],
            "unique_exercises": attempts["exercise_id"].nunique(),
            "work_modes": ", ".join(sorted(attempts["work_mode"].unique())),
            "source_values": source_values,
            "playlist_contexts": playlist_contexts,
            "max_modules_per_playlist_context": max_modules_per_playlist_context,
        }
    ]
)
display(attempt_summary)
display(attempts.head())

,attempt_rows,students,classrooms,modules,activities,unique_exercises,work_modes,source_values,playlist_contexts,max_modules_per_playlist_context
0,5590740,37894,3091,27,1238,18367,"playlist, zpdes",None,2791,12


,student_id,classroom_id,exercise_id,activity_id,module,created_at,data_correct,work_mode,playlist_id,success
2,4cb67a2d-19f9-4747-bef5-962dd1938d70,5d180572-a36a-453b-b197-b8d33271732c,2e4b566d-709e-4c8e-9724-cac8ff10ff37,7b724644-07be-40e4-a2fe-365d581504b3,Réapprentissage du sens des nombres,2024-02-27 16:09:11.525000+00:00,False,zpdes,053df3ec-5501-4ad8-9917-a935bcf76740,0.0
3,4cb67a2d-19f9-4747-bef5-962dd1938d70,5d180572-a36a-453b-b197-b8d33271732c,91d33267-99da-45c5-938b-7bb0f90d0537,d0d4d31a-b9ad-4502-94cc-b837f1b6f861,Réapprentissage du sens des nombres,2024-02-27 16:08:05.993000+00:00,False,zpdes,053df3ec-5501-4ad8-9917-a935bcf76740,0.0
4,4cb67a2d-19f9-4747-bef5-962dd1938d70,5d180572-a36a-453b-b197-b8d33271732c,3fd33fba-b331-47a3-9b69-1e0d1f9128f3,d0d4d31a-b9ad-4502-94cc-b837f1b6f861,Réapprentissage du sens des nombres,2024-02-27 16:06:49.965000+00:00,False,zpdes,053df3ec-5501-4ad8-9917-a935bcf76740,0.0
5,4cb67a2d-19f9-4747-bef5-962dd1938d70,5d180572-a36a-453b-b197-b8d33271732c,16efabbe-20fb-4291-9039-a33d78548d23,924165fb-0e16-456e-98f9-b52797eeb65e,Réapprentissage du sens des nombres,2024-02-27 16:09:31.514000+00:00,False,zpdes,053df3ec-5501-4ad8-9917-a935bcf76740,0.0
10,4cb67a2d-19f9-4747-bef5-962dd1938d70,5d180572-a36a-453b-b197-b8d33271732c,783d3179-56a9-427f-9a85-5ac8cb5ba959,924165fb-0e16-456e-98f9-b52797eeb65e,Réapprentissage du sens des nombres,2024-02-27 16:08:58.563000+00:00,True,zpdes,053df3ec-5501-4ad8-9917-a935bcf76740,1.0


## 4. Split the two student populations

The split uses all retained playlist/zpdes attempts:

- `exclusive_modes`: remove every student who appears in both modes.
- `both_modes`: keep only students who appear in both modes.

No global minimum number of exercises is imposed on students. Eligibility is applied later within each activity sequence.

In [4]:
populations = split_populations(attempts)

population_summary = pd.DataFrame(
    [
        {
            "population": population,
            "attempt_rows": len(frame),
            "students": frame["student_id"].nunique(),
            "classrooms": frame["classroom_id"].nunique(),
            "modules": frame["module"].nunique(),
            "activities": frame[["module", "activity_id"]].drop_duplicates().shape[0],
            "unique_exercises": frame["exercise_id"].nunique(),
            "playlist_rows": int((frame["work_mode"] == "playlist").sum()),
            "zpdes_rows": int((frame["work_mode"] == "zpdes").sum()),
        }
        for population, frame in populations.items()
    ]
)
display(population_summary)

,population,attempt_rows,students,classrooms,modules,activities,unique_exercises,playlist_rows,zpdes_rows
0,exclusive_modes,4121629,31603,3010,27,1238,18344,1016204,3105425
1,both_modes,1469111,6291,719,26,1223,17689,617355,851756


## 5. Build `mean_progress`

`mean_progress` is computed for each contiguous work-mode run within a sequence-specific activity. Playlist exercises are mapped through `config_mia.json` learning items to the same UUID-backed activities used by ZPDES, then qualified by playlist ID; the same activity in two playlists therefore produces two observations. ZPDES activities retain their raw UUID identifiers.

First, only the earliest retained playlist/zpdes attempt for each student–exercise pair is kept. Within each student/sequence/activity timeline, these attempts are sorted and split whenever work mode changes. Exercises from other activities may occur in between without breaking this activity timeline. The metric is computed separately for every resulting run:

`(success_rate_later_half - success_rate_first_half) * 100`

This means an estimate of `+5` is a 5 percentage-point increase in first-attempt success rate from the first half to the later half. Sequences with fewer than `MIN_ACTIVITY_EXERCISES` unique exercises are dropped. For odd sequence lengths, the middle exercise is excluded.

In [5]:
activity_by_population = {}

for population, frame in populations.items():
    activity_level = build_activity_level(
        frame,
        min_activity_exercises=MIN_ACTIVITY_EXERCISES,
        exercise_elo=exercise_elo,
    )
    activity_level["population"] = population
    activity_by_population[population] = activity_level

activity_summary = pd.DataFrame(
    [
        {
            "population": population,
            "activity_mode_rows": len(frame),
            "students": frame["student_id"].nunique(),
            "classrooms": frame["classroom_id"].nunique(),
            "modules": frame["module"].nunique(),
            "activities": frame[["module", "activity_id"]].drop_duplicates().shape[0],
            "mean_progress_mean": frame["mean_progress"].mean(),
            "mean_progress_sd": frame["mean_progress"].std(),
        }
        for population, frame in activity_by_population.items()
    ]
)

display(activity_summary)
display(pd.concat(activity_by_population.values(), ignore_index=True).head())

,population,activity_mode_rows,students,classrooms,modules,activities,mean_progress_mean,mean_progress_sd
0,exclusive_modes,351242,28036,2812,27,11868,14.152753,31.604001
1,both_modes,116771,6188,709,24,7458,11.056522,31.765563


,student_id,module,activity_id,activity_sequence_id,work_mode,success_rate_first,success_rate_later,success_rate_all,n_first_attempts,unique_exercises,classroom_id,mean_exercise_elo_first,elo_exercises_first,half_exercises_first,elo_exact_context_first,elo_exercise_fallback_first,mean_exercise_elo_later,elo_exercises_later,half_exercises_later,elo_exact_context_later,elo_exercise_fallback_later,mean_progress,population
0,00017460-1206-45ee-b1d3-393c72a45220,Améliorer la compréhension des textes,3190aa1d-5766-4524-bc6a-e8f53325a754,1,zpdes,0.500000,1.00,0.800000,5,5,132484cb-02b9-4cc2-974a-e38e485dd1f1,1389.990429,2,2,2,0,1535.777379,2,2,2,0,50.000000,exclusive_modes
1,00017460-1206-45ee-b1d3-393c72a45220,Améliorer la compréhension des textes,329a1d42-e738-4219-bf05-b787e85d1e69,1,zpdes,0.333333,1.00,0.666667,6,6,132484cb-02b9-4cc2-974a-e38e485dd1f1,1657.593523,3,3,3,0,1486.020396,3,3,3,0,66.666667,exclusive_modes
2,00017460-1206-45ee-b1d3-393c72a45220,Améliorer la compréhension des textes,380dbdeb-57f2-4f2a-8c54-c4ef0d044a4f,1,zpdes,0.750000,0.25,0.444444,9,9,132484cb-02b9-4cc2-974a-e38e485dd1f1,1534.120596,4,4,4,0,1503.893930,4,4,4,0,-50.000000,exclusive_modes
3,00017460-1206-45ee-b1d3-393c72a45220,Améliorer la compréhension des textes,4f8180ad-bcea-401a-be8b-ea06bc944699,1,zpdes,1.000000,1.00,1.000000,4,4,132484cb-02b9-4cc2-974a-e38e485dd1f1,1481.326117,2,2,2,0,1532.316799,2,2,2,0,0.000000,exclusive_modes
4,00017460-1206-45ee-b1d3-393c72a45220,Améliorer la compréhension des textes,5d20f168-a60b-11ed-afa1-0242ac120006,1,zpdes,1.000000,1.00,1.000000,4,4,132484cb-02b9-4cc2-974a-e38e485dd1f1,1535.114139,2,2,2,0,1300.798380,2,2,2,0,0.000000,exclusive_modes


## 6. Audit module loss across filters

This audit shows how many modules still have `playlist`, `zpdes`, or both modes after each filter stage.

There is no global student exercise threshold. The final stage reports coverage after selecting first student–exercise attempts and requiring at least four unique exercises within each contiguous activity/work-mode run.

In [6]:
def summarize_module_mode_coverage(frame: pd.DataFrame, stage: str) -> dict:
    if frame.empty:
        return {
            "stage": stage,
            "rows": 0,
            "students": 0,
            "modules_total": 0,
            "modules_with_playlist": 0,
            "modules_with_zpdes": 0,
            "modules_with_both_modes": 0,
            "playlist_rows": 0,
            "zpdes_rows": 0,
        }

    mode_sets = frame.groupby("module")["work_mode"].agg(lambda values: set(values))
    return {
        "stage": stage,
        "rows": len(frame),
        "students": frame["student_id"].nunique(),
        "modules_total": frame["module"].nunique(),
        "modules_with_playlist": int(mode_sets.map(lambda modes: "playlist" in modes).sum()),
        "modules_with_zpdes": int(mode_sets.map(lambda modes: "zpdes" in modes).sum()),
        "modules_with_both_modes": int(
            mode_sets.map(lambda modes: {"playlist", "zpdes"}.issubset(modes)).sum()
        ),
        "playlist_rows": int((frame["work_mode"] == "playlist").sum()),
        "zpdes_rows": int((frame["work_mode"] == "zpdes").sum()),
    }


def build_module_mode_detail(frame: pd.DataFrame, stage: str) -> pd.DataFrame:
    if frame.empty:
        return pd.DataFrame()

    aggregations = {
        "rows": ("work_mode", "size"),
        "students": ("student_id", "nunique"),
    }
    if "exercise_id" in frame.columns:
        aggregations["unique_exercises"] = ("exercise_id", "nunique")
    if "activity_id" in frame.columns:
        aggregations["activities"] = ("activity_id", "nunique")

    detail = frame.groupby(["module", "work_mode"], as_index=False).agg(**aggregations)
    detail.insert(0, "stage", stage)
    return detail


population_attempts_combined = pd.concat(populations.values(), ignore_index=True)
activity_combined = pd.concat(activity_by_population.values(), ignore_index=True)

audit_stages = [
    ("01 after work-mode filter, all playlist rows kept", attempts),
    ("02 after population split, no global exercise threshold, combined", population_attempts_combined),
    ("03 after contiguous first-attempt mean_progress + min 4 activity exercises, combined", activity_combined),
]

for population, frame in populations.items():
    audit_stages.append((f"02b after population split, {population}", frame))

for population, frame in activity_by_population.items():
    audit_stages.append((f"03b after contiguous first-attempt mean_progress, {population}", frame))

filter_audit_summary = pd.DataFrame(
    [summarize_module_mode_coverage(frame, stage) for stage, frame in audit_stages]
)
display(filter_audit_summary)

module_mode_audit = pd.concat(
    [build_module_mode_detail(frame, stage) for stage, frame in audit_stages],
    ignore_index=True,
)
display(module_mode_audit.sort_values(["stage", "module", "work_mode"]))

,stage,rows,students,modules_total,modules_with_playlist,modules_with_zpdes,modules_with_both_modes,playlist_rows,zpdes_rows
0,"01 after work-mode filter, all playlist rows kept",5590740,37894,27,25,27,25,1633559,3957181
1,"02 after population split, no global exercise threshold, combined",5590740,37894,27,25,27,25,1633559,3957181
2,"03 after contiguous first-attempt mean_progress + min 4 activity exercises, combined",468013,34224,27,24,27,24,118104,349909
3,"02b after population split, exclusive_modes",4121629,31603,27,25,27,25,1016204,3105425
4,"02b after population split, both_modes",1469111,6291,26,24,26,24,617355,851756
5,"03b after contiguous first-attempt mean_progress, exclusive_modes",351242,28036,27,24,27,24,74143,277099
6,"03b after contiguous first-attempt mean_progress, both_modes",116771,6188,24,24,24,24,43961,72810


,stage,module,work_mode,rows,students,unique_exercises,activities
0,"01 after work-mode filter, all playlist rows kept",Algorithmique et programmation,playlist,23256,901,255.0,28
1,"01 after work-mode filter, all playlist rows kept",Algorithmique et programmation,zpdes,46744,1965,260.0,28
2,"01 after work-mode filter, all playlist rows kept",Amélioration de la production écrite et de l'activité rédactionnelle (textes argumentatifs),playlist,14927,730,141.0,27
3,"01 after work-mode filter, all playlist rows kept",Amélioration de la production écrite et de l'activité rédactionnelle (textes argumentatifs),zpdes,21656,1503,83.0,15
4,"01 after work-mode filter, all playlist rows kept",Amélioration de la production écrite et de l'activité rédactionnelle (textes descriptifs),zpdes,353,41,17.0,3
...,...,...,...,...,...,...,...
303,"03b after contiguous first-attempt mean_progress, exclusive_modes",Syntaxe niveau 2,zpdes,7045,393,NaN,85
304,"03b after contiguous first-attempt mean_progress, exclusive_modes",Verbe niveau 1,playlist,2915,430,NaN,407
305,"03b after contiguous first-attempt mean_progress, exclusive_modes",Verbe niveau 1,zpdes,7103,1239,NaN,38
306,"03b after contiguous first-attempt mean_progress, exclusive_modes",Verbe niveau 2,playlist,1198,175,NaN,233


## 7. Compare first-half and later-half success levels

The progress-score model below compares changes, but it cannot show whether playlist and ZPDES sequences start from different success levels. This complementary model keeps two observations per eligible sequence: its first-half and later-half success rates.

Before fitting the model, the Elo table compares exercise difficulty in those same sequence halves. Each eligible sequence receives equal weight: first the mean exercise Elo is calculated within each sequence-half, then the mean and median of those sequence-level values are reported by work mode. Elo coverage and exact/fallback match counts are shown so the difficulty comparison can be audited.

`success_rate ~ work_mode * half + (1 | classroom_id) + (1 | student_id) + (1 | sequence_id)`

The `work_mode` coefficient estimates the adjusted ZPDES-versus-playlist difference in the first half. The interaction estimates the difference in progression between the two modes (a difference-in-differences). The sequence random intercept pairs the two halves of the same sequence. Results are expressed in percentage points and use the pooled `combined` population. As with the primary model, this is an observational comparison rather than a randomized causal estimate.

In [7]:
combined_activity_level = pd.concat(
    activity_by_population.values(),
    ignore_index=True,
).copy()
combined_activity_level["source_population"] = combined_activity_level["population"]
combined_activity_level["population"] = "combined"

model_activity_by_population = {
    **activity_by_population,
    "combined": combined_activity_level,
}

half_elo_summary = summarize_half_exercise_elo(combined_activity_level)
display(half_elo_summary.round(2))

half_elo_comparison = half_elo_summary.pivot(
    index="half",
    columns="work_mode",
    values=["mean_sequence_elo", "median_sequence_elo", "elo_coverage"],
)
display(half_elo_comparison.round(2))

half_descriptive_summary = (
    combined_activity_level.groupby("work_mode", as_index=False)
    .agg(
        sequences=("work_mode", "size"),
        students=("student_id", "nunique"),
        first_half_success=("success_rate_first", "mean"),
        later_half_success=("success_rate_later", "mean"),
    )
)
half_descriptive_summary[["first_half_success", "later_half_success"]] *= 100
half_descriptive_summary["change_points"] = (
    half_descriptive_summary["later_half_success"]
    - half_descriptive_summary["first_half_success"]
)
display(half_descriptive_summary.round(2))

half_success_model_summary = pd.DataFrame()
half_success_teacher_summary = pd.DataFrame()

if RUN_MODEL:
    half_success_result = fit_half_success_model(
        combined_activity_level,
        population="combined",
        maxiter=MAXITER,
    )
    half_success_model_summary = pd.DataFrame([half_success_result.__dict__])
    half_success_teacher_summary = half_success_model_summary[
        [
            "population",
            "status",
            "converged",
            "n_sequences",
            "n_students",
            "playlist_first_half",
            "playlist_later_half",
            "playlist_change",
            "zpdes_first_half",
            "zpdes_later_half",
            "zpdes_change",
            "initial_zpdes_vs_playlist",
            "initial_ci_low",
            "initial_ci_high",
            "initial_p_value",
            "difference_in_differences",
            "did_ci_low",
            "did_ci_high",
            "did_p_value",
        ]
    ].copy()
    half_success_teacher_summary["reportable"] = (
        half_success_teacher_summary["status"].eq("ok")
        & half_success_teacher_summary["converged"].eq(True)
    )
    display(half_success_teacher_summary.round(2))
    display(
        half_success_model_summary[
            [
                "model_specification",
                "optimizer",
                "scale",
                "log_likelihood",
                "random_classroom_var",
                "random_student_var",
                "random_sequence_var",
                "variance_components",
                "error",
            ]
        ]
    )
else:
    print("RUN_MODEL is False, so the half-level model was not fitted.")

,work_mode,half,sequences,sequences_with_elo,students,mean_sequence_elo,median_sequence_elo,elo_exercises,half_exercises,exact_context_rows,fallback_rows,elo_coverage
0,playlist,first,118104,118104,13667,1475.38,1478.35,627496,627496,627496,0,1.0
1,playlist,later,118104,118104,13667,1481.38,1489.70,627496,627496,627496,0,1.0
2,zpdes,first,349909,349909,25489,1521.96,1527.15,1176795,1176795,1176795,0,1.0
3,zpdes,later,349909,349909,25489,1508.56,1513.04,1176795,1176795,1176795,0,1.0


mean_sequence_elo          median_sequence_elo           \
work_mode          playlist    zpdes            playlist    zpdes   
half                                                                
first               1475.38  1521.96             1478.35  1527.15   
later               1481.38  1508.56             1489.70  1513.04   

          elo_coverage        
work_mode     playlist zpdes  
half                          
first              1.0   1.0  
later              1.0   1.0

,work_mode,sequences,students,first_half_success,later_half_success,change_points
0,playlist,118104,13667,70.04,71.89,1.84
1,zpdes,349909,25489,61.94,79.21,17.27


c:\Users\ocler\Documents\Académique\Inria\GAIMHE\Code\visu2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,population,status,converged,n_sequences,n_students,playlist_first_half,playlist_later_half,playlist_change,zpdes_first_half,zpdes_later_half,zpdes_change,initial_zpdes_vs_playlist,initial_ci_low,initial_ci_high,initial_p_value,difference_in_differences,did_ci_low,did_ci_high,did_p_value,reportable
0,combined,ok,True,468013,34224,72.1,74.07,1.97,59.79,77.11,17.32,-12.31,-12.66,-11.96,0.0,15.35,15.14,15.55,0.0,True


,model_specification,optimizer,scale,log_likelihood,random_classroom_var,random_student_var,random_sequence_var,variance_components,error
0,"Gaussian GPBoost half-level model; work mode by half fixed effects; random intercepts for classroom, student, and pa...",lbfgs,479.345222,-4.444171e+06,117.441602,109.526844,342.751325,classroom_id=117.442; student_id=109.527; sequence_id=342.751,None


## 8. Fit the mixed model

The primary model is:

`mean_progress ~ work_mode + (1 | classroom_id) + (1 | student_id)`

Activity is deliberately not included as an adjustment because ZPDES selects among activities. Activity selection is therefore part of the system being evaluated, rather than a pre-existing characteristic to hold constant.

GPBoost fits classroom and student as separate random intercepts. Student identifiers are globally unique, and each student belongs to exactly one classroom. The reported work-mode coefficient is one global `zpdes`-versus-`playlist` difference that includes the association operating through ZPDES activity selection. The optimizer uses L-BFGS.

The two source populations and their pooled union are fitted. The existing pooled population-interaction model and clustered-OLS sensitivity check retain their module fixed-effect specifications; they are separate secondary analyses.

Only rows marked `reportable = True` in the teacher summary should be interpreted. A non-converged optimizer can return numerical estimates, but those estimates remain provisional.

In [8]:
combined_composition = (
    combined_activity_level.groupby(
        ["work_mode", "source_population"],
        as_index=False,
    )
    .agg(
        activity_mode_rows=("mean_progress", "size"),
        students=("student_id", "nunique"),
    )
    .sort_values(["work_mode", "source_population"])
)
display(combined_composition)

fit_summaries = []
interaction_summary_table = pd.DataFrame()
sensitivity_summary = pd.DataFrame()

if RUN_MODEL:
    for population, activity_level in model_activity_by_population.items():
        fit_summaries.append(
            fit_mixed_model(
                activity_level,
                population=population,
                maxiter=MAXITER,
            )
        )

    interaction_summary = fit_population_interaction_model(
        combined_activity_level,
        maxiter=MAXITER,
    )
    interaction_summary_table = pd.DataFrame([interaction_summary.__dict__])

    sensitivity_summaries = [
        fit_clustered_ols_sensitivity(activity_level, population).__dict__
        for population, activity_level in model_activity_by_population.items()
    ]
    sensitivity_summary = pd.DataFrame(sensitivity_summaries)
else:
    print("RUN_MODEL is False, so only data preparation and plotting will run.")

model_summary = pd.DataFrame([summary.__dict__ for summary in fit_summaries])
display(model_summary)
display(interaction_summary_table)
display(sensitivity_summary)

,work_mode,source_population,activity_mode_rows,students
0,playlist,both_modes,43961,5529
1,playlist,exclusive_modes,74143,8138
2,zpdes,both_modes,72810,5591
3,zpdes,exclusive_modes,277099,19898


,population,status,n_rows,n_students,n_classrooms,n_modules,n_activities,model_specification,optimizer,intercept,intercept_std_error,intercept_p_value,intercept_ci_low,intercept_ci_high,estimate_zpdes_vs_playlist,std_error,p_value,ci_low,ci_high,playlist_adjusted_mean,zpdes_adjusted_mean,converged,scale,log_likelihood,random_student_var,random_classroom_var,variance_components,warning_count,warning_messages,error
0,exclusive_modes,ok,351242,28036,2812,27,11868,Gaussian GPBoost model; random intercepts for classroom and student,lbfgs,2.178879,0.204798,1.959229e-26,1.777474,2.580284,15.322975,0.235807,0.0,14.860794,15.785156,2.178879,17.501854,True,939.561907,-1.703344e+06,13.294048,10.393612,classroom_id=10.3936; student_id=13.294,0,None,None
1,both_modes,ok,116771,6188,709,24,7458,Gaussian GPBoost model; random intercepts for classroom and student,lbfgs,1.835271,0.215876,1.871567e-17,1.412154,2.258388,15.185734,0.211095,0.0,14.771987,15.599480,1.835271,17.021004,True,937.868226,-5.658303e+05,6.586033,8.202373,classroom_id=8.20237; student_id=6.58603,0,None,None
2,combined,ok,468013,34224,2910,27,14886,Gaussian GPBoost model; random intercepts for classroom and student,lbfgs,2.115720,0.148454,4.378073e-46,1.824750,2.406691,15.275894,0.157572,0.0,14.967053,15.584736,2.115720,17.391614,True,939.392778,-2.269166e+06,11.409584,9.961151,classroom_id=9.96115; student_id=11.4096,0,None,None


,status,n_rows,n_students,n_classrooms,n_modules,model_specification,optimizer,exclusive_playlist_change,exclusive_zpdes_change,exclusive_zpdes_vs_playlist,exclusive_std_error,exclusive_p_value,exclusive_ci_low,exclusive_ci_high,both_playlist_change,both_zpdes_change,both_zpdes_vs_playlist,both_std_error,both_p_value,both_ci_low,both_ci_high,interaction_both_minus_exclusive,interaction_std_error,interaction_p_value,interaction_ci_low,interaction_ci_high,converged,random_student_var,random_classroom_var,scale,warning_count,warning_messages,error
0,ok,468013,34224,2910,27,classroom random intercept; student random intercept nested within classroom; module fixed effects,lbfgs,2.138456,17.347792,15.209336,0.217482,0.0,14.783079,15.635594,1.870618,16.985821,15.115203,0.213508,0.0,14.696727,15.533679,-0.094134,0.290323,0.745758,-0.663157,0.47489,True,10.410085,7.828003,936.888884,0,None,None


,population,status,n_rows,n_students,n_classrooms,n_modules,model_specification,playlist_adjusted_mean,zpdes_adjusted_mean,estimate_zpdes_vs_playlist,std_error,p_value,ci_low,ci_high,error
0,exclusive_modes,ok,351242,28036,2812,27,OLS with module fixed effects and classroom-clustered standard errors,2.032757,17.395684,15.362926,0.227495,0.0,14.917044,15.808809,None
1,both_modes,ok,116771,6188,709,24,OLS with module fixed effects and classroom-clustered standard errors,1.331024,16.928554,15.597530,0.311934,0.0,14.986150,16.208910,None
2,combined,ok,468013,34224,2910,27,OLS with module fixed effects and classroom-clustered standard errors,1.748310,17.306328,15.558018,0.190083,0.0,15.185463,15.930573,None


## 9. Read the model output

The coefficient table below prints the fixed-effect estimates in a format closer to a regression output:

- `Intercept`: estimated playlist mean progress when the classroom and student random effects are at their population-average value of zero. It is a model estimate, not the raw playlist average.
- `work_mode: zpdes vs playlist`: estimated difference in mean progress points for `zpdes` compared with `playlist`.
- `std_error`, `ci_low`, `ci_high`, and `p_value`: uncertainty around each fixed-effect estimate.

The teacher table uses population-level fixed-effect predictions: the playlist estimate is the intercept, and the zpdes estimate is the intercept plus the work-mode coefficient. `random_student_var` and `random_classroom_var` describe random-intercept variation, not fixed effects.

In the interaction table, `interaction_both_minus_exclusive` is the difference between the two population-specific zpdes-versus-playlist effects. A value near zero means the associations are similar in size.

This is observational data, so the work-mode coefficient should be read as an adjusted association, not a causal effect.

In [9]:
if not model_summary.empty:
    fixed_effect_rows = []

    for _, row in model_summary.iterrows():
        fixed_effect_rows.append(
            {
                "population": row["population"],
                "status": row["status"],
                "converged": row["converged"],
                "term": "Intercept (playlist, population-level)",
                "estimate": row["intercept"],
                "std_error": row["intercept_std_error"],
                "ci_low": row["intercept_ci_low"],
                "ci_high": row["intercept_ci_high"],
                "p_value": row["intercept_p_value"],
                "interpretation": "Model-estimated playlist mean_progress at zero classroom and student random effects",
            }
        )
        fixed_effect_rows.append(
            {
                "population": row["population"],
                "status": row["status"],
                "converged": row["converged"],
                "term": "work_mode: zpdes vs playlist",
                "estimate": row["estimate_zpdes_vs_playlist"],
                "std_error": row["std_error"],
                "ci_low": row["ci_low"],
                "ci_high": row["ci_high"],
                "p_value": row["p_value"],
                "interpretation": "Difference in mean_progress for zpdes relative to playlist",
            }
        )

    fixed_effects = pd.DataFrame(fixed_effect_rows)
    display(fixed_effects)

    teacher_summary = model_summary.loc[
        model_summary["status"].isin(["ok", "not_converged"]),
        [
            "population",
            "status",
            "converged",
            "n_rows",
            "n_students",
            "playlist_adjusted_mean",
            "zpdes_adjusted_mean",
            "estimate_zpdes_vs_playlist",
            "ci_low",
            "ci_high",
            "p_value",
        ],
    ].copy()
    teacher_summary["reportable"] = (
        teacher_summary["status"].eq("ok")
        & teacher_summary["converged"].eq(True)
    )
    teacher_summary = teacher_summary.rename(
        columns={
            "playlist_adjusted_mean": "playlist_change_points",
            "zpdes_adjusted_mean": "zpdes_change_points",
            "estimate_zpdes_vs_playlist": "difference_points",
            "ci_low": "difference_ci_low",
            "ci_high": "difference_ci_high",
            "p_value": "difference_p_value",
        }
    )[
        [
            "population",
            "status",
            "converged",
            "reportable",
            "n_rows",
            "n_students",
            "playlist_change_points",
            "zpdes_change_points",
            "difference_points",
            "difference_ci_low",
            "difference_ci_high",
            "difference_p_value",
        ]
    ]
    display(teacher_summary.round(2))
    if not teacher_summary["reportable"].all():
        print("Do not report rows where reportable=False; inspect convergence diagnostics first.")

    diagnostics = model_summary[
        [
            "population",
            "status",
            "model_specification",
            "optimizer",
            "n_rows",
            "n_students",
            "n_classrooms",
            "n_modules",
            "n_activities",
            "converged",
            "scale",
            "log_likelihood",
            "random_student_var",
            "random_classroom_var",
            "variance_components",
            "warning_count",
            "warning_messages",
            "error",
        ]
    ]
    display(diagnostics)

    if not sensitivity_summary.empty:
        robustness_summary = model_summary[
            ["population", "estimate_zpdes_vs_playlist", "ci_low", "ci_high"]
        ].merge(
            sensitivity_summary[
                ["population", "estimate_zpdes_vs_playlist", "ci_low", "ci_high"]
            ],
            on="population",
            suffixes=("_mixed_model", "_clustered_ols"),
        )
        display(robustness_summary.round(2))

,population,status,converged,term,estimate,std_error,ci_low,ci_high,p_value,interpretation
0,exclusive_modes,ok,True,"Intercept (playlist, population-level)",2.178879,0.204798,1.777474,2.580284,1.959229e-26,Model-estimated playlist mean_progress at zero classroom and student random effects
1,exclusive_modes,ok,True,work_mode: zpdes vs playlist,15.322975,0.235807,14.860794,15.785156,0.000000e+00,Difference in mean_progress for zpdes relative to playlist
2,both_modes,ok,True,"Intercept (playlist, population-level)",1.835271,0.215876,1.412154,2.258388,1.871567e-17,Model-estimated playlist mean_progress at zero classroom and student random effects
3,both_modes,ok,True,work_mode: zpdes vs playlist,15.185734,0.211095,14.771987,15.599480,0.000000e+00,Difference in mean_progress for zpdes relative to playlist
4,combined,ok,True,"Intercept (playlist, population-level)",2.115720,0.148454,1.824750,2.406691,4.378073e-46,Model-estimated playlist mean_progress at zero classroom and student random effects
5,combined,ok,True,work_mode: zpdes vs playlist,15.275894,0.157572,14.967053,15.584736,0.000000e+00,Difference in mean_progress for zpdes relative to playlist


,population,status,converged,reportable,n_rows,n_students,playlist_change_points,zpdes_change_points,difference_points,difference_ci_low,difference_ci_high,difference_p_value
0,exclusive_modes,ok,True,True,351242,28036,2.18,17.50,15.32,14.86,15.79,0.0
1,both_modes,ok,True,True,116771,6188,1.84,17.02,15.19,14.77,15.60,0.0
2,combined,ok,True,True,468013,34224,2.12,17.39,15.28,14.97,15.58,0.0


,population,status,model_specification,optimizer,n_rows,n_students,n_classrooms,n_modules,n_activities,converged,scale,log_likelihood,random_student_var,random_classroom_var,variance_components,warning_count,warning_messages,error
0,exclusive_modes,ok,Gaussian GPBoost model; random intercepts for classroom and student,lbfgs,351242,28036,2812,27,11868,True,939.561907,-1.703344e+06,13.294048,10.393612,classroom_id=10.3936; student_id=13.294,0,None,None
1,both_modes,ok,Gaussian GPBoost model; random intercepts for classroom and student,lbfgs,116771,6188,709,24,7458,True,937.868226,-5.658303e+05,6.586033,8.202373,classroom_id=8.20237; student_id=6.58603,0,None,None
2,combined,ok,Gaussian GPBoost model; random intercepts for classroom and student,lbfgs,468013,34224,2910,27,14886,True,939.392778,-2.269166e+06,11.409584,9.961151,classroom_id=9.96115; student_id=11.4096,0,None,None


,population,estimate_zpdes_vs_playlist_mixed_model,ci_low_mixed_model,ci_high_mixed_model,estimate_zpdes_vs_playlist_clustered_ols,ci_low_clustered_ols,ci_high_clustered_ols
0,exclusive_modes,15.32,14.86,15.79,15.36,14.92,15.81
1,both_modes,15.19,14.77,15.60,15.60,14.99,16.21
2,combined,15.28,14.97,15.58,15.56,15.19,15.93


## 10. Prepare the top-module forest plot

The forest plot is descriptive. It shows module-level mean progress and confidence intervals by work mode only: one `playlist` point and one `zpdes` point per module. Here, both module selection and plotted estimates pool the filtered `exclusive_modes` and `both_modes` populations.

Selection is performed after the playlist filter and activity-sequence eligibility rule. A module must contain both work modes in the pooled data, then modules are ranked by total raw attempt rows (`playlist` + `zpdes`) across both source populations. The plot contains up to `TOP_N_MODULES` modules; it contains fewer when fewer modules have both modes. There is currently no minimum number of students required separately in each work mode. This means a module can rank highly because of its `zpdes` volume even when its playlist group is small.

The usage table exposes the raw attempt-row, student, and exercise counts separately for playlist and zpdes. In `plot_table`, `n_rows` counts eligible contiguous activity/work-mode progress-run records built from first attempts, not raw attempts.

It does not replace the mixed model above. The model gives the global adjusted `zpdes` vs `playlist` estimate; the forest plot helps inspect where the largest-volume modules sit descriptively.

In [10]:
usage, plot_table = build_top_module_plot_table(
    populations,
    activity_by_population,
    top_n_modules=TOP_N_MODULES,
    require_both_work_modes=not ALLOW_SINGLE_MODE_TOP_MODULES,
    plot_by_population=PLOT_BY_POPULATION,
    forest_population=FOREST_POPULATION,
)

if not usage.empty:
    selected_count = usage.loc[usage["selected_for_plot"], "module"].nunique()
    eligible_count = usage.loc[usage["eligible_for_plot"], "module"].nunique()
    print(
        f"Forest population: {FOREST_POPULATION or 'combined'}; "
        f"selected {selected_count} of {TOP_N_MODULES} requested modules "
        f"from {eligible_count} modules containing both work modes."
    )

display(
    usage.sort_values(["module", "population"])
    if usage.empty
    else usage.sort_values(
        ["selected_for_plot", "usage_rank", "attempt_rows"],
        ascending=[False, True, False],
        na_position="last",
    ).head(30)
)
display(plot_table)

Forest population: combined; selected 5 of 5 requested modules from 24 modules containing both work modes.


,population,module,attempt_rows,students,unique_exercises,playlist_attempt_rows,playlist_students,playlist_unique_exercises,zpdes_attempt_rows,zpdes_students,zpdes_unique_exercises,eligible_for_plot,usage_rank,selected_for_plot
22,exclusive_modes,Réapprentissage du sens des nombres,1177892,10486,1062,253366,2396,1045,924526,8090,1062,True,1,True
48,both_modes,Réapprentissage du sens des nombres,403229,3138,1062,129775,1401,1045,273454,2512,1062,True,1,True
23,exclusive_modes,Syntaxe niveau 1,414734,4091,697,85268,1037,694,329466,3054,697,True,2,True
49,both_modes,Syntaxe niveau 1,131568,1184,695,59570,744,674,71998,682,695,True,2,True
9,exclusive_modes,Comprendre les notions de proportion et de fraction,366624,3829,902,75980,1134,902,290644,2695,902,True,3,True
35,both_modes,Comprendre les notions de proportion et de fraction,147727,1634,902,40894,866,902,106833,981,902,True,3,True
19,exclusive_modes,Orthographe niveau 1,291340,3535,509,52597,579,499,238743,2956,509,True,4,True
45,both_modes,Orthographe niveau 1,102313,1056,509,58039,676,499,44274,655,509,True,4,True
5,exclusive_modes,Améliorer la compréhension des textes,281787,2834,630,69474,649,624,212313,2185,630,True,5,True
31,both_modes,Améliorer la compréhension des textes,76991,650,630,25500,245,602,51491,462,630,True,5,True


,population,module,work_mode,estimate,std,n_rows,students,activities,se,ci_low,ci_high,module_order
6,combined,Réapprentissage du sens des nombres,playlist,3.116446,23.848829,25029,3095,2624,0.150746,2.820984,3.411908,0
7,combined,Réapprentissage du sens des nombres,zpdes,19.039792,30.082824,100528,9823,70,0.094880,18.853827,19.225757,0
8,combined,Syntaxe niveau 1,playlist,0.152940,27.721177,11883,1709,1078,0.254301,-0.345491,0.651370,1
9,combined,Syntaxe niveau 1,zpdes,18.883179,31.414590,35015,3414,60,0.167882,18.554130,19.212228,1
2,combined,Comprendre les notions de proportion et de fraction,playlist,2.133413,30.202616,9663,1436,1311,0.307248,1.531208,2.735618,2
3,combined,Comprendre les notions de proportion et de fraction,zpdes,19.227247,32.030981,35100,3262,68,0.170969,18.892148,19.562346,2
4,combined,Orthographe niveau 1,playlist,-1.048729,25.887000,8213,1196,835,0.285648,-1.608599,-0.488860,3
5,combined,Orthographe niveau 1,zpdes,15.154101,35.617276,21316,3231,38,0.243954,14.675951,15.632251,3
0,combined,Améliorer la compréhension des textes,playlist,-0.512347,25.191717,4727,828,669,0.366408,-1.230507,0.205813,4
1,combined,Améliorer la compréhension des textes,zpdes,18.089794,32.717809,21553,2399,33,0.222859,17.652990,18.526598,4


## 11. Display the forest plot

Each point is the average `mean_progress` for one module and one work mode. Horizontal bars are approximate 95% confidence intervals around that descriptive mean.

In [11]:
FOREST_X_RANGE = (-15, 25)
MODULE_LABEL_WRAP_WIDTH = 32
FOREST_EXPORT_WIDTH = 1200
displayed_module_count = plot_table['module'].nunique() if not plot_table.empty else 0
FOREST_FIGURE_HEIGHT = max(750, 150 * displayed_module_count)
FOREST_PLOT_CONFIG = {
    'toImageButtonOptions': {
        'format': 'svg',
        'filename': 'work_mode_progress_top_modules_forest',
        'width': FOREST_EXPORT_WIDTH,
        'height': FOREST_FIGURE_HEIGHT,
        'scale': 1,
    },
    'displaylogo': False,
    'responsive': True,
}

fig = build_forest_figure(plot_table)

if fig is None:
    print("No forest plot was created. Try lowering filters or set ALLOW_SINGLE_MODE_TOP_MODULES = True.")
else:
    module_label_annotations = []
    for tick_value, module in zip(
        fig.layout.yaxis.tickvals,
        fig.layout.yaxis.ticktext,
        strict=True,
    ):
        label_lines = textwrap.wrap(
            str(module).replace('-', '\N{NON-BREAKING HYPHEN}'),
            width=MODULE_LABEL_WRAP_WIDTH,
            break_long_words=False,
            break_on_hyphens=False,
        )
        label_center = (len(label_lines) - 1) / 2
        for line_index, label_line in enumerate(label_lines):
            module_label_annotations.append(
                {
                    'xref': 'paper',
                    'yref': 'y',
                    'x': -0.025,
                    'y': tick_value,
                    'text': label_line,
                    'showarrow': False,
                    'xanchor': 'right',
                    'yanchor': 'middle',
                    'yshift': (label_center - line_index) * 17,
                    'font': {'size': 12, 'color': '#2A3F5F'},
                }
            )
    fig.update_xaxes(range=list(FOREST_X_RANGE))
    fig.update_yaxes(showticklabels=False, title_text='')
    fig.update_layout(
        height=FOREST_FIGURE_HEIGHT,
        margin={'l': 280, 'r': 40, 't': 70, 'b': 60},
        annotations=[
            *list(fig.layout.annotations or ()),
            *module_label_annotations,
        ],
    )
    fig.show(config=FOREST_PLOT_CONFIG)

## 12. Display separate maths and French forest plots

These two additional descriptive plots apply the same eligibility, ranking, estimation, and confidence-interval logic as the global forest plot. Ranking is recomputed within each subject: up to five mathematics modules and up to five French modules are selected by pooled raw attempt volume.

In [12]:
# Additional top-five forest plots by subject.
import json

SUBJECT_TOP_N_MODULES = 5
MATHS_MODULE_CODES = {f'M{module_number}' for module_number in range(101, 109)}

with MODULE_CONFIG_JSON.open(encoding='utf-8') as module_config_file:
    configured_modules = json.load(module_config_file)['config']['module']

module_subject = {}
for module_key, module_metadata in configured_modules.items():
    module_code = str(module_metadata.get('code') or f'M{module_key}')
    module_title = module_metadata.get('title') or {}
    module_name = module_title.get('short') or module_title.get('long') or module_code
    module_subject[module_name] = (
        'Math\u00e9matiques' if module_code in MATHS_MODULE_CODES else 'Fran\u00e7ais'
    )

observed_modules = set().union(
    *(set(frame['module'].dropna().unique()) for frame in populations.values())
)
unclassified_modules = observed_modules.difference(module_subject)
if unclassified_modules:
    raise ValueError(
        'Modules without a maths/French catalogue classification: '
        f'{sorted(unclassified_modules)}'
    )


def _style_subject_forest_figure(subject_figure, subject_title, subject_plot_table):
    displayed_count = subject_plot_table['module'].nunique()
    figure_height = max(750, 150 * displayed_count)
    label_annotations = []
    for tick_value, module in zip(
        subject_figure.layout.yaxis.tickvals,
        subject_figure.layout.yaxis.ticktext,
        strict=True,
    ):
        label_lines = textwrap.wrap(
            str(module).replace('-', '\N{NON-BREAKING HYPHEN}'),
            width=MODULE_LABEL_WRAP_WIDTH,
            break_long_words=False,
            break_on_hyphens=False,
        )
        label_center = (len(label_lines) - 1) / 2
        for line_index, label_line in enumerate(label_lines):
            label_annotations.append(
                {
                    'xref': 'paper',
                    'yref': 'y',
                    'x': -0.025,
                    'y': tick_value,
                    'text': label_line,
                    'showarrow': False,
                    'xanchor': 'right',
                    'yanchor': 'middle',
                    'yshift': (label_center - line_index) * 17,
                    'font': {'size': 12, 'color': '#2A3F5F'},
                }
            )
    subject_figure.update_xaxes(range=list(FOREST_X_RANGE))
    subject_figure.update_yaxes(showticklabels=False, title_text='')
    subject_figure.update_layout(
        title=subject_title,
        title_x=0.5,
        height=figure_height,
        margin={'l': 280, 'r': 40, 't': 85, 'b': 60},
        annotations=[
            *list(subject_figure.layout.annotations or ()),
            *label_annotations,
        ],
    )
    return figure_height


subject_forest_outputs = {}
for subject_name, subject_slug in (
    ('Math\u00e9matiques', 'maths'),
    ('Fran\u00e7ais', 'francais'),
):
    subject_modules = {
        module for module, subject in module_subject.items() if subject == subject_name
    }
    subject_populations = {
        population: frame[frame['module'].isin(subject_modules)].copy()
        for population, frame in populations.items()
    }
    subject_activities = {
        population: frame[frame['module'].isin(subject_modules)].copy()
        for population, frame in activity_by_population.items()
    }
    subject_usage, subject_plot_table = build_top_module_plot_table(
        subject_populations,
        subject_activities,
        top_n_modules=SUBJECT_TOP_N_MODULES,
        require_both_work_modes=not ALLOW_SINGLE_MODE_TOP_MODULES,
        plot_by_population=PLOT_BY_POPULATION,
        forest_population=FOREST_POPULATION,
    )
    subject_figure = build_forest_figure(subject_plot_table)
    subject_forest_outputs[subject_slug] = {
        'usage': subject_usage,
        'plot_table': subject_plot_table,
        'figure': subject_figure,
    }

    selected_count = (
        subject_usage.loc[subject_usage['selected_for_plot'], 'module'].nunique()
        if not subject_usage.empty
        else 0
    )
    eligible_count = (
        subject_usage.loc[subject_usage['eligible_for_plot'], 'module'].nunique()
        if not subject_usage.empty
        else 0
    )
    print(
        f'{subject_name}: selected {selected_count} of '
        f'{SUBJECT_TOP_N_MODULES} requested modules from {eligible_count} eligible modules.'
    )
    if subject_figure is None:
        print(f'No {subject_name.lower()} forest plot could be created.')
        continue

    subject_height = _style_subject_forest_figure(
        subject_figure,
        f'Top 5 des modules de {subject_name.lower()} \u2014 progr\u00e8s moyen par work mode',
        subject_plot_table,
    )
    subject_config = {
        'toImageButtonOptions': {
            'format': 'svg',
            'filename': f'work_mode_progress_top_5_{subject_slug}_forest',
            'width': FOREST_EXPORT_WIDTH,
            'height': subject_height,
            'scale': 1,
        },
        'displaylogo': False,
        'responsive': True,
    }
    subject_figure.show(config=subject_config)

Mathématiques: selected 5 of 5 requested modules from 8 eligible modules.


Français: selected 5 of 5 requested modules from 16 eligible modules.


## 13. Save outputs

The same CSV and HTML outputs as the script are written to `OUTPUT_DIR` for sharing or later inspection.

In [13]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

half_elo_summary.to_csv(
    OUTPUT_DIR / "work_mode_half_exercise_elo_summary.csv",
    index=False,
    encoding="utf-8-sig",
)
half_descriptive_summary.to_csv(
    OUTPUT_DIR / "work_mode_half_success_descriptive.csv",
    index=False,
    encoding="utf-8-sig",
)
if not half_success_model_summary.empty:
    half_success_model_summary.to_csv(
        OUTPUT_DIR / "work_mode_half_success_model_summary.csv",
        index=False,
        encoding="utf-8-sig",
    )
if not half_success_teacher_summary.empty:
    half_success_teacher_summary.to_csv(
        OUTPUT_DIR / "work_mode_half_success_teacher_summary.csv",
        index=False,
        encoding="utf-8-sig",
    )

model_summary.to_csv(
    OUTPUT_DIR / "work_mode_progress_model_summary.csv",
    index=False,
    encoding="utf-8-sig",
)
if "teacher_summary" in globals():
    teacher_summary.to_csv(
        OUTPUT_DIR / "work_mode_progress_teacher_summary.csv",
        index=False,
        encoding="utf-8-sig",
    )
if not interaction_summary_table.empty:
    interaction_summary_table.to_csv(
        OUTPUT_DIR / "work_mode_progress_population_interaction.csv",
        index=False,
        encoding="utf-8-sig",
    )
if not sensitivity_summary.empty:
    sensitivity_summary.to_csv(
        OUTPUT_DIR / "work_mode_progress_sensitivity.csv",
        index=False,
        encoding="utf-8-sig",
    )
if "robustness_summary" in globals():
    robustness_summary.to_csv(
        OUTPUT_DIR / "work_mode_progress_robustness_comparison.csv",
        index=False,
        encoding="utf-8-sig",
    )
usage.to_csv(
    OUTPUT_DIR / "work_mode_progress_module_usage.csv",
    index=False,
    encoding="utf-8-sig",
)
plot_table.to_csv(
    OUTPUT_DIR / "work_mode_progress_top_modules_forest_data.csv",
    index=False,
    encoding="utf-8-sig",
)
if fig is not None:
    fig.write_html(
        OUTPUT_DIR / "work_mode_progress_top_modules_forest.html",
        include_plotlyjs="cdn",
        config=FOREST_PLOT_CONFIG,
    )

print(f"Saved outputs to: {OUTPUT_DIR}")

Saved outputs to: c:\Users\ocler\Documents\Académique\Inria\GAIMHE\Code\visu2\artifacts\regression_mia_notebook
